## Prompt Engineering in LangSmith

### Import environment variables

In [1]:
import os, httpx
from dotenv import load_dotenv

load_dotenv("/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/.env", override=True)

CA_BUNDLE = "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/ca-bundle.pem"
os.environ["SSL_CERT_FILE"] = CA_BUNDLE
os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "eli5-prompting"

http_client = httpx.Client(verify=CA_BUNDLE)

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

### Pull in Prompt from Prompthub

In [2]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate
from langsmith.utils import LangSmithNotFoundError

client = Client(api_key=LANGSMITH_API_KEY)

# Define the prompt template
eli5_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. 

Your task is to take a complex question and context information, then provide a clear, simple explanation using:
- Simple words and concepts
- Analogies and examples from everyday life
- Short sentences
- Engaging and friendly tone

Keep your explanation concise but complete.

Question: {question}

Context: {context}

Please explain this in simple terms that a 5-year-old would understand:
""")])

# Try to pull the prompt, if it doesn't exist, push it first
try:
    print("Trying to pull existing prompt...")
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled existing prompt from LangSmith")
except LangSmithNotFoundError:
    print("❌ Prompt not found. Creating and pushing new prompt...")
    
    # Push the prompt to LangSmith
    client.push_prompt(
        "eli5-concise",
        object=eli5_prompt_template,
        description="A prompt for explaining complex topics in simple terms that a 5-year-old could understand"
    )
    print("✅ Successfully pushed prompt to LangSmith")
    
    # Now pull the prompt back
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled the newly created prompt")

Trying to pull existing prompt...
❌ Prompt not found. Creating and pushing new prompt...
✅ Successfully pushed prompt to LangSmith
✅ Successfully pulled the newly created prompt


### Setup AI Application

Let's first setup our web search tool, as usual

In [3]:
# Initialize web search tool

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1)

/var/folders/wr/xj2yx28s7xvg8trnrb4nc5_w0000gn/T/ipykernel_55292/1343233806.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1)


Let's now create our application, same as in the tracing module. This time, our prompt is the one pulled from PromptHub

In [4]:
from groq import Groq
from langsmith import traceable

# Create Application
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable(run_type="llm")
def explain(question, context):
    # Format the prompt with the question and context
    messages = prompt.format_messages(question=question, context=context)
    
    # Call the Groq API with the formatted messages
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": msg.type, "content": msg.content} for msg in messages],
        temperature=0.7
    )
    
    return response.choices[0].message.content

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer

### Test Application

In [5]:
question = "what is complexity economics?"
print(eli5(question))

Imagine you're playing with lots of different toys and friends in a big room. That's like the economy. It's a big system with many parts, like people, stores, and governments, all interacting with each other.

Now, some people think the economy is like a simple puzzle, where everything fits together in a straight line. But complexity economics says, "No, it's more like a big, messy playground!" Everything is connected and can change in surprising ways.

It's like when you're playing with blocks, and you add one more block, and then the whole tower falls down! You didn't expect that to happen, but it did. That's kind of like what happens in the economy. Things can change suddenly, and it's hard to predict what will happen next.

So, complexity economics tries to understand how all these different parts work together and how they can change and surprise us. It's like trying to figure out how to build a really cool fort with all your friends, and you're not sure what it will look like unt